In [ ]:
import os
import sys
import argparse
import pandas as pd
import torch
import anndata as ad
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from DeepRUOT.losses import OT_loss1
from DeepRUOT.utils import (
    generate_steps, load_and_merge_config,
    SchrodingerBridgeConditionalFlowMatcher,
    generate_state_trajectory, get_batch, get_batch_size
)
from DeepRUOT.train import train_un1_reduce, train_all
from DeepRUOT.models import FNet_interaction, scoreNet2
from DeepRUOT.constants import DATA_DIR, RES_DIR
from DeepRUOT.exp import setup_exp

In [ ]:
config_path = '../config/mosta_config.yaml'

# Load and merge configuration
config = load_and_merge_config(config_path)

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, config['data']['file_path']))
df = df.iloc[:, :config['data']['dim'] + 1]
#df = df[df.iloc[:,1] > 0.4]
device = torch.device('cpu')
exp_dir, logger = setup_exp(
            RES_DIR, 
            config, 
            config['exp']['name']
        )
dim = config['data']['dim']

In [ ]:
model_config = config['model']
        
f_net = FNet_interaction(
            in_out_dim=model_config['in_out_dim'],
            hidden_dim=model_config['hidden_dim'],
            n_hiddens=model_config['n_hiddens'],
            activation=model_config['activation'],
            use_spatial = True, 
            num_heads = 8,
            thre = 0.06,
            num_layers = 1,

        ).to(device)

sf2m_score_model = scoreNet2(
    in_out_dim=model_config['in_out_dim'],
    hidden_dim=model_config['score_hidden_dim'],
    activation=model_config['activation']
).float().to(device)

In [ ]:
f_net.load_state_dict(torch.load(os.path.join(exp_dir, 'model_final'),map_location=torch.device('cpu')))
f_net.to(device)
sf2m_score_model.load_state_dict(torch.load(os.path.join(exp_dir, 'score_model'),map_location=torch.device('cpu')))
sf2m_score_model.to(device)

In [ ]:
import scanpy as sc

# 加载 h5ad 文件
adata = sc.read("../spatial_data/Mouse_embryo_all_stage.h5ad")

# 查看数据的基本信息
print(adata)

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance

# 假设 adata 是您的原始 AnnData 对象
# 获取所有唯一的批次名称
batch_names = adata.obs['timepoint'].cat.categories

# 创建字典存储每个批次的 AnnData 对象，确保是实际对象
adata_dict = {}
for batch in batch_names:
    adata_dict[batch] = adata[adata.obs['timepoint'] == batch].copy()

# 定义预处理函数
def preprocess_adata(adata):
    #sc.pp.normalize_total(adata, target_sum=1e4)
    #sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata = adata[:, adata.var.highly_variable]
    return adata

# 定义空间坐标缩放函数
def scale_spatial_coords(adata):
    spatial_coords = adata.obsm['spatial']
    x_min, x_max = spatial_coords[:, 0].min(), spatial_coords[:, 0].max()
    y_min, y_max = spatial_coords[:, 1].min(), spatial_coords[:, 1].max()
    x_range = x_max - x_min
    spatial_coords[:, 0] = (spatial_coords[:, 0] - x_min) / x_range
    scale_factor = 1 / x_range
    spatial_coords[:, 1] = (spatial_coords[:, 1] - y_min) * scale_factor
    adata.obsm['spatial'] = spatial_coords # 更新修改后的坐标
    return adata

# 定义绘图函数
def plot_spatial(adata, batch_name):
    spatial_coords = adata.obsm['spatial']
    annotations = adata.obs['Annotation']
    annotation_colors = adata.uns['Annotation_colors']
    category_to_color = dict(zip(annotations.cat.categories, annotation_colors))
    colors = annotations.map(category_to_color)
    plt.figure(figsize=(10, 8))
    plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=colors, s=10, alpha=1)
    plt.title(f'Spatial Visualization for {batch_name}')
    plt.xlabel('Scaled X')
    plt.ylabel('Scaled Y')
    plt.show()


def remove_outliers(adata, radius=0.05, threshold=5):
    # 获取空间坐标
    spatial_coords = adata.obsm['spatial']
    # 计算所有数据点之间的距离矩阵
    dist_matrix = distance.cdist(spatial_coords, spatial_coords)
    # 计算每个点的邻居数量（减去自身）
    neighbors = np.sum(dist_matrix < radius, axis=1) - 1
    # 创建掩码：保留邻居数量大于等于阈值的点
    mask = neighbors >= threshold
    # 应用掩码，删除离群点
    adata = adata[mask]
    return adata

# 自动化处理每个批次
for batch in batch_names:
    adata_batch = adata_dict[batch].copy()
    # adata_batch = preprocess_adata(adata_batch)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    # adata_batch = remove_outliers(adata_batch, radius=0.05, threshold=5)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    adata_dict[batch]=adata_batch.copy()

import numpy as np
import pandas as pd

# 初始化标签列表
labels_list = []
T = 5
batch_indices = [3, 4, 5, 6]

for t, batch_idx in enumerate(batch_indices):
    adata_t = adata_dict[batch_names[batch_idx]]
    labels_t = adata_t.obs['annotation'].values  # 获取当前时间点的 Annotation
    labels_list.append(labels_t)

# 合并所有标签
all_labels = np.concatenate(labels_list)

# 读取 CSV 文件
df_new = pd.read_csv('../data/mosta_four_time.csv')

# 添加 Annotation 列
df_new['Annotation'] = all_labels
# 获取 Annotation 的类别和颜色
if pd.api.types.is_categorical_dtype(adata.obs['annotation']):
    categories = adata.obs['annotation'].cat.categories
else:
    categories = adata.obs['annotation'].unique()  # 如果不是 categorical 类型

colors = adata.uns['annotation_colors']

# 创建标签到颜色的映射
label_to_color = dict(zip(categories, colors))

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import kaleido
import plotly.io as pio

def plot_nature_methods_sankey(predicted_labels_list, time_keys, label_to_color, 
                               target_tissue=None, title="Cell Fate Dynamics"):
    
    links = []
    
    # 1. 构建 Links
    for t in range(len(time_keys) - 1):
        src_labels = predicted_labels_list[t]
        tgt_labels = predicted_labels_list[t+1]
        min_len = min(len(src_labels), len(tgt_labels))
        
        df = pd.DataFrame({'source': src_labels[:min_len], 'target': tgt_labels[:min_len]})
        
        if target_tissue:
            df = df[(df['target'] == target_tissue) | (df['source'] == target_tissue)]
        
        if len(df) == 0: continue

        counts = df.groupby(['source', 'target']).size().reset_index(name='value')
        # === 每个 target 只保留收到的 top3 transitions ===
        counts = (
            counts
            .sort_values(["target", "value"], ascending=[True, False])
            .groupby("target")
            .head(3)
            .reset_index(drop=True)
        )
        counts['source_id'] = counts['source'].astype(str) + f"__T{t}"
        counts['target_id'] = counts['target'].astype(str) + f"__T{t+1}"
        links.append(counts)
    
    if not links:
        print("无数据连线")
        return

    all_links = pd.concat(links, axis=0)
    
    # 2. 构建 Nodes
    unique_node_ids = list(pd.concat([all_links['source_id'], all_links['target_id']]).unique())
    node_map = {node: i for i, node in enumerate(unique_node_ids)}
    
    node_colors = []
    node_labels = []
    node_x = []
    node_y = []
    
    for node_id in unique_node_ids:
        real_name, t_suffix = node_id.split('__T')
        t_idx = int(t_suffix)
        
        node_labels.append(real_name)
        
        # 修复：获取并清洗颜色 (节点通常不透明)
        raw_c = label_to_color.get(real_name, "#888888")
        node_colors.append(to_valid_color(raw_c, default_alpha=1.0))
        
        x_pos = 0.05 + (t_idx / (len(time_keys) - 1)) * 0.9
        node_x.append(x_pos)
        node_y.append(None)
        
    # 3. 连线颜色
    source_indices = all_links['source_id'].map(node_map)
    target_indices = all_links['target_id'].map(node_map)
    
    link_colors = []
    for src_id in all_links['source_id']:
        real_name = src_id.split('__T')[0]
        # 修复：获取并清洗颜色 (连线半透明)
        raw_c = label_to_color.get(real_name, "#888888")
        link_colors.append(to_valid_color(raw_c, default_alpha=0.4))
        
    fig = go.Figure(data=[go.Sankey(
        arrangement = "snap",
        node = dict(
            pad = 15, thickness = 20,
            line = dict(color = "black", width = 0.5),
            label = node_labels,
            color = node_colors, # 已修复
            x = node_x, y = node_y,
            hovertemplate = 'Type: %{label}<br>Count: %{value}<extra></extra>'
        ),
        link = dict(
            source = source_indices,
            target = target_indices,
            value = all_links['value'],
            color = link_colors, # 已修复
            hovertemplate = '%{source.label} → %{target.label}<br>%{value}<extra></extra>'
        )
    )])

    fig.update_layout(
        width=1600,      # 画布宽
        height=1000,     # 画布高
        title=dict(text=title, x=0.5, xanchor="center")
    )
    
    for i, tk in enumerate(time_keys):
        fig.add_annotation(
            x=0.0 + (i / (len(time_keys) - 1)), y=-0.1,
            text=tk, showarrow=False,
            font=dict(size=14, weight="bold", color="black")
        )

    fig.show()

    fig.write_image(
        "nature_methods_sankey.pdf",
        width=1600,
        height=1000,
        format="pdf",
        engine="kaleido"
    )

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# 假设数据已经加载到 DataFrame 中
# df = pd.read_csv('your_data.csv')  # 替换为你的数据文件路径

# 步骤 1：准备数据
# 提取特征和标签
X = df_new.iloc[:,:-1].values
y = df_new['Annotation'].values

# 对标签进行编码
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(y_encoded[:100])
# 划分训练集和测试集（80% 训练，20% 测试）
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# 转换为 PyTorch 张量
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# 步骤 2：构建 MLP 模型
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # 第一隐藏层
        self.relu = nn.LeakyReLU()                          # 激活函数
        self.fc2 = nn.Linear(hidden_size, hidden_size) # 第二隐藏层
        self.fc3 = nn.Linear(hidden_size, num_classes) # 输出层
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

input_size = 53  # 特征数量
hidden_size = 256 # 隐藏层神经元数量
num_classes = len(label_encoder.classes_)  # 类别数量
model = MLP(input_size, hidden_size, num_classes)

# 步骤 3：训练模型
criterion = nn.CrossEntropyLoss()  # 交叉熵损失
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam 优化器

In [ ]:
import random
import numpy as np
device = 'cpu'
df = pd.read_csv(os.path.join(DATA_DIR, config['data']['file_path']))
df = df.iloc[:, :config['data']['dim'] + 1]

from DeepRUOT.utils import euler_sdeint
from DeepRUOT.interaction import cal_interaction
import joblib
class SDE(torch.nn.Module):
    noise_type = "diagonal"
    sde_type = "ito"

    def __init__(self, ode_drift, g, score, interaction, input_size=(3, 32, 32), sigma=1.0):
        super().__init__()
        self.drift = ode_drift
        self.score = score
        self.input_size = input_size
        self.sigma = sigma
        self.interaction = interaction
        self.g_net = g

    # Drift
    def f(self, t, y):
        z, lnw = y
        with torch.no_grad():
            drift=self.drift(t, z)
            dlnw = self.g_net(t, z)
            net_forces = cal_interaction(z, lnw, self.interaction, t, m=512)
        num = z.shape[0]
        t = t.expand(num, 1)  # 保持 t 的梯度信息并扩展其形状
        return (drift+net_forces, dlnw) #+net_forces

    # Diffusion
    def g(self, t, y):
        return torch.ones_like(y)*self.sigma
    

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from DeepRUOT.utils import euler_sdeint

# ============================================================================
# 1. 准备时间轴 (Time Definition)
# ============================================================================
# 定义全部时间点
all_time_keys = ['E11.5', 'E12.5', 'E13.5', 'E14.0', 'E14.5', 'E15.5', 'E16.5', 'E17.5']

# 定义相对于 E12.5 (t=0) 的时间值
# 向前推为负，向后推为正
time_map = {
    'E11.5': -1.0, # Backward
    'E12.5':  0.0, # Start (t0)
    'E13.5':  1.0,
    'E14.0':  1.5, # Interpolation
    'E14.5':  2.0,
    'E15.5':  3.0,
    'E16.5':  4.0,
    'E17.5':  5.0
}

# 分离出正向和负向的时间点张量
# Backward: 从 0 推到 -1
ts_backward = torch.tensor([0.0, -1.0], dtype=torch.float32).to(device)

# Forward: 从 0 推到 5.0 (包含插值点)
ts_forward = torch.tensor([0.0, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0], dtype=torch.float32).to(device)

# ============================================================================
# 2. 准备初始状态 x0 (取自 E12.5)
# ============================================================================
# 假设 df 中 'samples' 列对应的值：E12.5 对应 0 (或其他标识，请确保取到 E12.5 的数据)
# 这里的 0 指的是您原始数据里 E12.5 对应的 sample index
data_t0_df = df[df['samples'] == 0] # 请确保这里选中的是 E12.5 的数据

data_t0 = torch.tensor(data_t0_df.iloc[:, 1:config['data']['dim']+1].values, dtype=torch.float32)
data_t0 = data_t0.requires_grad_()

# 采样 10000 个粒子
if data_t0.shape[0] > 10000:
    indices = torch.randperm(data_t0.shape[0])[:10000]
    x0_subset = data_t0[indices].to(device)
else:
    x0_subset = data_t0.to(device)

lnw0 = torch.log(torch.ones(x0_subset.shape[0], 1) / x0_subset.shape[0]).to(device)
initial_state = (x0_subset, lnw0)

# ============================================================================
# 3. 双向 SDE 积分 (Two-pass Integration)
# ============================================================================
# 定义 SDE (保持不变)
sde = SDE(f_net.v_net, 
          f_net.g_net, 
          sf2m_score_model, 
          f_net.interaction_net, 
          input_size=(30,), 
          sigma=0)

print("正在计算向后推演 (E12.5 -> E11.5)...")
# 注意 dt 为负数
sde_traj_back, _ = euler_sdeint(sde, initial_state, dt=-0.05, ts=ts_backward)
# sde_traj_back 形状: [2, N, Dim], 包含 [t=0, t=-1]

print("正在计算向前推演 (E12.5 -> E17.5)...")
# 注意 dt 为正数
sde_traj_fwd, _ = euler_sdeint(sde, initial_state, dt=0.05, ts=ts_forward)
# sde_traj_fwd 形状: [7, N, Dim], 包含 [t=0, t=1, t=1.5, ...]

# ============================================================================
# 4. 拼接轨迹 (Stitching)
# ============================================================================
# 我们需要按时间顺序拼起来: E11.5 -> E12.5 -> ... -> E17.5

# 1. 取出 E11.5 的数据 (backward 结果的索引 1，因为索引 0 是 t=0)
traj_E11_5 = sde_traj_back[1:2] # shape (1, N, Dim)

# 2. 取出 E12.5 及之后的数据 (forward 的全部)
traj_rest = sde_traj_fwd      # shape (7, N, Dim)

# 3. 在第 0 维拼接
sde_point_all = torch.cat([traj_E11_5, traj_rest], dim=0)
# 最终形状: (8, N, Dim) -> 完美对应 8 个时间点

# 转换为 Numpy 列表结构以便后续处理
sde_point_np = sde_point_all.detach().cpu().numpy()
sde_point_list = sde_point_np.tolist()
sde_point_array = np.array(sde_point_list, dtype=object)

print(f"全时间段模拟完成。轨迹形状: {sde_point_all.shape}")

# ============================================================================
# 5. 预测与分类 (Prediction & Smoothing)
# ============================================================================

import numpy as np
import torch
from sklearn.neighbors import KNeighborsClassifier

# ==============================================================================
# 0. 辅助函数：补全概率矩阵 (处理 predict_proba 缺类问题)
# ==============================================================================
def to_full_proba(proba_curr, classes_found, n_classes):
    """
    将 KNN 输出的部分概率矩阵扩展为全概率矩阵。
    proba_curr: (N, n_found)
    classes_found: 当前批次里存在的类别索引 (e.g., [0, 2, 5])
    n_classes: 总类别数 (C)
    """
    n_samples = proba_curr.shape[0]
    proba_full = np.zeros((n_samples, n_classes), dtype=np.float32)
    
    # 将存在的类别的概率填入对应列
    for i, col_idx in enumerate(classes_found):
        proba_full[:, col_idx] = proba_curr[:, i]
        
    return proba_full

# ==============================================================================
# 1. 准备工作
# ==============================================================================
# 加载保存的参数
model.load_state_dict(torch.load(exp_dir + '/mlp_classifier.pth', map_location=torch.device('cpu')))
model.eval()
model.to(device)

# 获取总类别数 C
C = len(label_encoder.classes_)
print(f"总类别数 C: {C}")

predicted_labels_list_all = []

# 如果您想在 E16.5/E17.5 解决 Brain 消失的问题，建议结合 E15.5 的参考库
# 这里为了防止逻辑冲突，我先只用您提供的“空间自平滑”，但保留了 E15.5 参考库的接口
last_real_time_val = 3.0 # E15.5
# 建立 E15.5 参考库 (用于防止外推崩坏，可选)
ref_mask = df_new['samples'] == 3 # 假设3代表E15.5
if ref_mask.sum() > 0:
    ref_X = df_new.loc[ref_mask].iloc[:, 1:3].values
    ref_y = label_encoder.transform(df_new.loc[ref_mask, 'Annotation'].values)
    knn_ref = KNeighborsClassifier(n_neighbors=50).fit(ref_X, ref_y)
else:
    knn_ref = None

# ==============================================================================
# 2. 循环预测 (结合 MLP + 空间 KNN 微调)
# ==============================================================================
time_vals_ordered = sorted(time_map.values())

for i in range(len(sde_point_array)):
    # 1. 准备数据
    t_val = time_vals_ordered[i]
    traj_t = np.array(sde_point_array[i], dtype=np.float64)
    
    # 修复可能的 NaN
    if np.isnan(traj_t).any(): traj_t = np.nan_to_num(traj_t)
    
    traj_t_tensor = torch.tensor(traj_t).float().to(device)
    n_samples = traj_t.shape[0]
    
    # 时间截断 (防止 MLP 对 E17.5 这种大数值过敏)
    # 如果 t_val > 3.0，我们输入 3.0，但位置使用 SDE 推演后的新位置
    mlp_t = min(t_val, 3.0) 
    if mlp_t < 0: mlp_t = 0 # 防止 E11.5 的 -1 输入
    
    # 构造输入: Time + Features
    samples_t = mlp_t * torch.ones((n_samples, 1)).to(device)
    input_t = torch.cat((samples_t, traj_t_tensor[:, :52]), dim=1)
    
    # --- A. MLP 预测 (Base Prediction) ---
    with torch.no_grad():
        logits = model(input_t)
        probs_mlp = torch.softmax(logits, dim=1).cpu().numpy() # (N, C)
        hard_pred_idx = probs_mlp.argmax(axis=1) # 初步硬标签索引
        
    # --- B. 空间 KNN 微调 (Spatial Fine-tuning) ---
    # 这就是您提供的代码逻辑：用 MLP 的结果训练一个临时的空间 KNN
    coords = input_t[:, 1:3].cpu().numpy() # 取 x, y
    
    # 策略判断：
    # 1. 对于内插时间点 (E11.5 - E15.5): 使用 "自平滑" (Self-KNN) -> 让图像更干净
    # 2. 对于外推时间点 (E16.5 - E17.5): 建议使用 "参考平滑" (Ref-KNN) -> 找回 Brain
    
    use_reference_correction = (t_val > 3.0) and (knn_ref is not None)
    
    if use_reference_correction:
        # 【外推阶段策略】: 强行参考 E15.5 的分布，防止 Brain 消失
        # 即使 MLP 预测全错了，只要位置在 Brain 区域，KNN_Ref 就会把它救回来
        print(f"  时间 {t_val}: 使用 E15.5 参考库修正 (Ref-KNN)")
        probs_final = knn_ref.predict_proba(coords) # 这里的 probs 直接就是 (N, C)
        
    else:
        # 【内插阶段策略】: 使用您提供的自平滑逻辑
        # print(f"  时间 {t_val}: 使用空间自平滑 (Self-KNN)")
        knn_self = KNeighborsClassifier(n_neighbors=61) # 您设定的 61
        knn_self.fit(coords, hard_pred_idx) # Fit on MLP prediction
        
        if hasattr(knn_self, "predict_proba"):
            proba_knn_curr = knn_self.predict_proba(coords)
            # 关键：补全概率矩阵维度
            probs_final = to_full_proba(proba_knn_curr, knn_self.classes_, C)
        else:
            refined_labels_idx = knn_self.predict(coords)
            probs_final = np.eye(C)[refined_labels_idx]

    # --- C. 最终决策 ---
    # 此时 probs_final 已经是平滑后的概率了
    final_pred_idx = probs_final.argmax(axis=1)
    
    # 转回字符串标签
    final_labels = label_encoder.inverse_transform(final_pred_idx)
    predicted_labels_list_all.append(final_labels)
    
    # 监控 Brain 数量
    brain_count = np.sum(final_labels == 'Brain') # 假设标签叫 Brain
    print(f"Time {t_val:>4}: Brain count = {brain_count}/{n_samples}")



In [ ]:
import pickle

# 保存到pkl文件
with open('mosta_predicted_labels_list_all_multilayer_communication.pkl', 'wb') as f:
    pickle.dump(predicted_labels_list_all, f)
    
print("预测列表生成完毕。")

In [ ]:
# 加载pkl文件
import pickle
with open('mosta_predicted_labels_list_all_multilayer_communication.pkl', 'rb') as f:
    predicted_labels_list_all = pickle.load(f)

In [ ]:
predicted_labels_list_all

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import colorsys

import plotly.graph_objects as go
import numpy as np
import pandas as pd

def to_valid_color(color_str, default_alpha=1.0):
    """
    将任意颜色字符串强制转换为 Plotly 可接受的格式 (rgba 或 6位 hex)。
    解决 '#bf024fff' 这种 8位 hex 报错的问题。
    """
    if not isinstance(color_str, str):
        return "rgba(136, 136, 136, 1)"
        
    color_str = color_str.strip()
    
    # 如果是 8位 Hex (#RRGGBBAA)
    if color_str.startswith('#') and len(color_str) == 9:
        r = int(color_str[1:3], 16)
        g = int(color_str[3:5], 16)
        b = int(color_str[5:7], 16)
        a = int(color_str[7:9], 16) / 255.0
        return f"rgba({r}, {g}, {b}, {a:.3f})"
    
    # 如果是 6位 Hex (#RRGGBB)，根据需要添加透明度
    elif color_str.startswith('#') and len(color_str) == 7:
        if default_alpha < 1.0:
            r = int(color_str[1:3], 16)
            g = int(color_str[3:5], 16)
            b = int(color_str[5:7], 16)
            return f"rgba({r}, {g}, {b}, {default_alpha})"
        else:
            return color_str # 6位 Hex 是合法的
            
    # 如果已经是 rgba 或 rgb，直接返回
    elif color_str.startswith('rgb'):
        return color_str
        
    # 其他情况（如颜色名），直接返回，如果报错说明颜色名不对
    return color_str

In [ ]:
# 2. 画 2D Nature Methods 风格图 (纯细胞命运)
fig_2d = plot_nature_methods_sankey(
    predicted_labels_list=predicted_labels_list_all,
    time_keys=['E11.5','E12.5', 'E13.5', 'E14.0', 'E14.5', 'E15.5','E16.5','E17.5'],
    label_to_color=label_to_color,
    title="Cell Fate Transitions"
)